In [12]:
import joblib
from sklearn.pipeline import Pipeline
import os
import pandas as pd
from pathlib import Path
import numpy as np

MODEL_PATH = r"D:\H2O2_predict\model_xgb.joblib"

In [ ]:
obj = joblib.load(MODEL_PATH)

print("=" * 60)
print("Loaded object type:")
print(type(obj))
print("=" * 60)

print(obj)
print("=" * 60)

if isinstance(obj, Pipeline):
    print("This joblib contains a Pipeline.")
    print("Pipeline steps:", list(obj.named_steps.keys()))
    if "preprocessor" in obj.named_steps:
        print("Preprocessor is included.")
    if "model" in obj.named_steps:
        print("Model is included.")
else:
    print("This joblib does NOT contain a sklearn Pipeline.")

Loaded object type:
<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['X1', 'X3', 'X4', 'X5', 'X6',
                                                   'X7', 'X8', 'X9', 'X10',
                                                   'X11', 'X12', 'X13', 'X14',
                                                   'X15', 'X16', 'X17', 'X18',
                                                   'X19', 'X20']),
                                                 ('cat',
                                                  Pipeline(steps=[('im

In [5]:
TARGET_DATA_PATH = r"D:\H2O2_predict\2_descriptor\Data_collect_Au_processed.xlsx"
OUT_DIR = r"D:\H2O2_predict\result"

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 70)
print("Load trained XGB model and predict new data")
print("=" * 70)

# ===== load model =====
model_xgb = joblib.load(MODEL_PATH)

# ===== read target/new data =====
df_target = pd.read_excel(TARGET_DATA_PATH)

# ===== extract feature columns =====
# current table: from 3rd column to 22nd column corresponds to old X1-X20
X_target = df_target.iloc[:, 2:22].copy()

if X_target.shape[1] != 20:
    raise ValueError(f"Expected 20 feature columns from column 3 to 22, got {X_target.shape[1]}")

# temporary rename to match training feature names
X_target.columns = [f"X{i}" for i in range(1, 21)]

# ===== predict =====
pred_xgb = model_xgb.predict(X_target)

# ===== save results =====
result_df = df_target.copy()
result_df["pred_xgb"] = pred_xgb

result_df.to_csv(os.path.join(OUT_DIR, "target_predictions_xgb.csv"), index=False, encoding="utf-8-sig")

print("\nPrediction finished.")
print("Saved to:", OUT_DIR)
print("=" * 70)

Load trained XGB model and predict new data

Prediction finished.
Saved to: D:\H2O2_predict\result


In [14]:
MODEL_PATH = r"D:\H2O2_predict\model_xgb.joblib"
INPUT_DIR = r"D:\H2O2_predict\new"
OUT_DIR = r"D:\H2O2_predict\new"

os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 70)
print("Batch load trained XGB model and predict Excel files")
print("=" * 70)

model_xgb = joblib.load(MODEL_PATH)

excel_files = []
for ext in ("*.xlsx", "*.xls"):
    excel_files.extend(Path(INPUT_DIR).glob(ext))

if not excel_files:
    raise FileNotFoundError(f"No Excel files found in: {INPUT_DIR}")

for file_path in excel_files:
    print(f"Processing: {file_path.name}")

    df = pd.read_excel(file_path)

    if df.shape[1] < 22:
        print(f"  Skipped: {file_path.name} (only {df.shape[1]} columns, need at least 22)")
        continue

    # 第3~22列 -> X1~X20
    X_target = df.iloc[:, 2:22].copy()
    X_target.columns = [f"X{i}" for i in range(1, 21)]

    # 将常见占位符替换为缺失值
    X_target = X_target.replace(["-", "--", "NA", "N/A", "na", ""], np.nan)

    # 训练时 X2 是类别列，其余列按数值处理
    num_cols = [f"X{i}" for i in range(1, 21) if f"X{i}" != "X2"]

    for col in num_cols:
        X_target[col] = pd.to_numeric(X_target[col], errors="coerce")

    # 如果希望 X2 也去掉前后空格，可加这一句
    X_target["X2"] = X_target["X2"].astype(str).str.strip()
    X_target["X2"] = X_target["X2"].replace(["nan", "None", ""], np.nan)

    pred_xgb = model_xgb.predict(X_target)

    result_df = pd.DataFrame({
        "Index": df.iloc[:, 1],
        "Name": df.iloc[:, 0],
        "pred_barrier": pred_xgb
    })

    out_path = os.path.join(OUT_DIR, f"{file_path.stem}_pred.xlsx")
    result_df.to_excel(out_path, index=False)

    print(f"  Saved: {out_path}")

print("=" * 70)
print("All done.")
print("Saved to:", OUT_DIR)
print("=" * 70)

Batch load trained XGB model and predict Excel files
Processing: DAC_SL_descriptor.xlsx
  Saved: D:\H2O2_predict\new\DAC_SL_descriptor_pred.xlsx
All done.
Saved to: D:\H2O2_predict\new
